In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model


class CustomCNN(Model):
    """
    CNN personnalisé pour la classification CIFAR-10.
    Architecture : 3 blocs Conv2D + MaxPooling → MLP + Dropout → Softmax
    """

    def __init__(self, num_classes: int = 10, dropout_rate: float = 0.4):
        super(CustomCNN, self).__init__()

        # ── Data Augmentation (intégrée au modèle) ──────────────────────────
        self.augment = tf.keras.Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
        ], name="data_augmentation")

        # ── Bloc 1 : Conv2D 32 filtres ───────────────────────────────────────
        self.conv1 = layers.Conv2D(32, (3, 3), padding="same", activation="relu",
                                   kernel_initializer="he_uniform", name="conv1")
        self.bn1   = layers.BatchNormalization(name="bn1")
        self.pool1 = layers.MaxPooling2D((2, 2), name="pool1")

        # ── Bloc 2 : Conv2D 64 filtres ───────────────────────────────────────
        self.conv2 = layers.Conv2D(64, (3, 3), padding="same", activation="relu",
                                   kernel_initializer="he_uniform", name="conv2")
        self.bn2   = layers.BatchNormalization(name="bn2")
        self.pool2 = layers.MaxPooling2D((2, 2), name="pool2")

        # ── Bloc 3 : Conv2D 128 filtres ──────────────────────────────────────
        self.conv3 = layers.Conv2D(128, (3, 3), padding="valid", activation="relu",
                                   kernel_initializer="he_uniform", name="conv3")
        self.bn3   = layers.BatchNormalization(name="bn3")

        # ── Classificateur MLP ───────────────────────────────────────────────
        self.flatten  = layers.Flatten(name="flatten")
        self.dense1   = layers.Dense(256, activation="relu",
                                     kernel_initializer="he_uniform", name="dense1")
        self.dropout  = layers.Dropout(dropout_rate, name="dropout")
        self.output_layer = layers.Dense(num_classes, activation="softmax",
                                         name="output")

    def call(self, inputs, training: bool = False):
        # Augmentation uniquement pendant l'entraînement
        x = self.augment(inputs, training=training)

        # Blocs convolutifs
        x = self.pool1(self.bn1(self.conv1(x), training=training))
        x = self.pool2(self.bn2(self.conv2(x), training=training))
        x = self.bn3(self.conv3(x), training=training)

        # MLP
        x = self.flatten(x)
        x = self.dense1(x)
        x = self.dropout(x, training=training)
        return self.output_layer(x)


def build_cnn(num_classes: int = 10) -> CustomCNN:
    """
    Instancie et retourne le modèle CustomCNN.
    Appel model.build() pour forcer la création des poids avant model.summary().
    """
    model = CustomCNN(num_classes=num_classes)
    model.build(input_shape=(None, 32, 32, 3))
    return model


# ── Test rapide ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    model = build_cnn()
    model.summary()


Model: "custom_cnn"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 data_augmentation (Sequent  (None, 32, 32, 3)         0         
 ial)                                                            
                                                                 
 conv1 (Conv2D)              multiple                  896       
                                                                 
 bn1 (BatchNormalization)    multiple                  128       
                                                                 
 pool1 (MaxPooling2D)        multiple                  0         
                                                                 
 conv2 (Conv2D)              multiple                  18496     
                                                                 
 bn2 (BatchNormalization)    multiple                  256       
                                                        

In [2]:
"""
models/rnn_model.py
Définition de l'architecture LSTM pour la prédiction de série temporelle (T+1).
Compatible avec tout dataset normalisé via MinMaxScaler.
"""

import tensorflow as tf
from tensorflow.keras import layers, Model


class LSTMForecaster(Model):
    """
    Réseau LSTM pour la prédiction univariée / multivariée au pas T+1.

    Paramètres
    ----------
    units_1 : int   — Nombre d'unités dans la 1ʳᵉ couche LSTM
    units_2 : int   — Nombre d'unités dans la 2ᵉ couche LSTM
    dropout_rate : float — Taux de dropout entre les couches
    output_size  : int  — Nombre de valeurs à prédire (1 pour univarié)
    """

    def __init__(self,
                 units_1: int = 128,
                 units_2: int = 64,
                 dropout_rate: float = 0.2,
                 output_size: int = 1):
        super(LSTMForecaster, self).__init__()

        # ── Couche 1 LSTM : return_sequences=True → transmet la séquence ────
        self.lstm1 = layers.LSTM(
            units_1,
            return_sequences=True,   # Nécessaire pour empiler une 2ᵉ couche
            kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            name="lstm1"
        )
        self.dropout1 = layers.Dropout(dropout_rate, name="dropout1")

        # ── Couche 2 LSTM : return_sequences=False → ne garde que l'état final
        self.lstm2 = layers.LSTM(
            units_2,
            return_sequences=False,  # ⚠ Exigé par le sujet pour prédire T+1
            kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            name="lstm2"
        )
        self.dropout2 = layers.Dropout(dropout_rate, name="dropout2")

        # ── Couche Dense finale : prédiction T+1 ────────────────────────────
        self.output_layer = layers.Dense(output_size, name="output")

    def call(self, inputs, training: bool = False):
        x = self.lstm1(inputs, training=training)
        x = self.dropout1(x, training=training)
        x = self.lstm2(x, training=training)
        x = self.dropout2(x, training=training)
        return self.output_layer(x)


def build_lstm(sequence_length: int = 24,
               n_features: int = 1,
               output_size: int = 1) -> LSTMForecaster:
    """
    Instancie et construit le modèle LSTM.

    Paramètres
    ----------
    sequence_length : longueur de la fenêtre glissante
    n_features      : nombre de variables (1 = univarié)
    output_size     : nombre de valeurs à prédire
    """
    model = LSTMForecaster(output_size=output_size)
    model.build(input_shape=(None, sequence_length, n_features))
    return model


# ── Test rapide ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    model = build_lstm(sequence_length=24, n_features=1)
    model.summary()


Model: "lstm_forecaster"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm1 (LSTM)                multiple                  66560     
                                                                 
 dropout1 (Dropout)          multiple                  0         
                                                                 
 lstm2 (LSTM)                multiple                  49408     
                                                                 
 dropout2 (Dropout)          multiple                  0         
                                                                 
 output (Dense)              multiple                  65        
                                                                 
Total params: 116033 (453.25 KB)
Trainable params: 116033 (453.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
